In [ ]:
# ==============================================================================
# INSTALACIÓN DE DEPENDENCIAS E INICIO DE SESIÓN PYSPARK (2.1.2)
# Proyecto EV2 Big Data - Integrantes: Ricardo Cayupel & Lorenzo Aguila
# ==============================================================================
!pip install pyspark boto3 s3fs matplotlib seaborn pandas pyyaml -q

import os
import urllib.request
from getpass import getpass
import boto3
from botocore.exceptions import ClientError
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    Imputer,
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler
)
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# Inicializar motor distribuido Apache Spark con compresión Snappy y Apache Arrow
spark = SparkSession.builder \
    .appName("EV2_BigData_AWS_Pipeline_Cayupel_Aguila") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.sql.parquet.compression.codec", "snappy") \
    .getOrCreate()

print(f"Clúster PySpark iniciado correctamente. Versión de Spark: {spark.version}")

In [ ]:
# ==============================================================================
# GOBIERNO IAM Y APROVISIONAMIENTO AUTOMÁTICO DE S3 (2.1.2)
# ==============================================================================
print("Ingresa las credenciales programáticas del usuario IAM 'bigdata-user':")
AWS_ACCESS_KEY = getpass("1. AWS Access Key ID: ").strip()
AWS_SECRET_KEY = getpass("2. AWS Secret Access Key: ").strip()
AWS_REGION = "us-east-1"
S3_BUCKET = "ev2-bigdata-cayupel-aguila"

# Cliente Boto3 autenticado con IAM
s3_client = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=AWS_REGION
)

# Verificar si el bucket existe; si no existe, lo crea automáticamente en us-east-1
try:
    s3_client.head_bucket(Bucket=S3_BUCKET)
    print(f"Bucket existente detectado: s3://{S3_BUCKET}/")
except ClientError:
    print(f"Creando bucket Data Lake '{S3_BUCKET}' en AWS S3 ({AWS_REGION})...")
    s3_client.create_bucket(Bucket=S3_BUCKET)
    print(f"Bucket s3://{S3_BUCKET}/ creado exitosamente.")

# Crear las 3 zonas/carpetas requeridas en el Data Lake S3
for folder in ["raw/", "curated/", "athena-results/"]:
    s3_client.put_object(Bucket=S3_BUCKET, Key=folder)

print("Zonas del Data Lake ('raw/', 'curated/', 'athena-results/') listas en Amazon S3.")

In [ ]:
# ==============================================================================
# CAPA DE INGESTA BATCH HACIA DATA LAKE AMAZON S3 (2.1.1 y 2.1.2)
# ==============================================================================
os.makedirs("datalake/raw", exist_ok=True)
os.makedirs("datalake/curated", exist_ok=True)

url_dataset = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
local_raw_path = "datalake/raw/yellow_tripdata_2024-01.parquet"

if not os.path.exists(local_raw_path):
    print("Descargando dataset masivo NYC Yellow Taxi (Enero 2024)...")
    urllib.request.urlretrieve(url_dataset, local_raw_path)
    print("Dataset descargado en el entorno temporal.")

print(f"☁️ Subiendo datos brutos a Zona Bronce: s3://{S3_BUCKET}/raw/ ...")
s3_client.upload_file(local_raw_path, S3_BUCKET, "raw/yellow_tripdata_2024-01.parquet")
print("Ingesta en Zona Bronce (S3) completada.")

# Lectura distribuida con PySpark
df_raw = spark.read.parquet(local_raw_path)
print(f"Total de registros en el dataset original: {df_raw.count():,}")

# Muestra representativa del 5% (~148,000 viajes) para procesamiento ágil en Free Tier
df_sample = df_raw.sample(withReplacement=False, fraction=0.05, seed=42)
print(f"Registros seleccionados para el pipeline distribuido: {df_sample.count():,}")

In [ ]:
# ==============================================================================
# PROCESAMIENTO ETL, FORMATO PARQUET Y PARTICIONAMIENTO (2.1.2)
# ==============================================================================
print("Ejecutando limpieza ETL distribuida e ingeniería de variables en PySpark...")

df_clean = df_sample.withColumn(
    "trip_duration_min",
    F.round((F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0, 2)
).withColumn(
    "pickup_hour", F.hour("tpep_pickup_datetime")
).withColumn(
    "pickup_day_of_week", F.dayofweek("tpep_pickup_datetime")
).withColumn(
    "payment_type_str",
    F.when(F.col("payment_type") == 1, "Credit_Card")
     .when(F.col("payment_type") == 2, "Cash")
     .otherwise("Other")
).filter(
    (F.col("trip_distance") >= 0.2) & (F.col("trip_distance") <= 80.0) &
    (F.col("fare_amount") >= 3.0) & (F.col("fare_amount") <= 250.0) &
    (F.col("trip_duration_min") >= 1.0) & (F.col("trip_duration_min") <= 180.0)
).select(
    F.col("VendorID").cast("int"),
    F.col("passenger_count").cast("double"),
    F.col("trip_distance").cast("double"),
    F.col("RatecodeID").cast("double"),
    F.col("pickup_hour").cast("int"),
    F.col("pickup_day_of_week").cast("int"),
    F.col("trip_duration_min").cast("double"),
    F.col("tip_amount").cast("double"),
    F.col("fare_amount").cast("double"),
    F.col("total_amount").cast("double"),
    F.col("payment_type_str")
)

# Guardar en formato PARQUET con compresión Snappy particionado por 'payment_type_str'
curated_local_dir = "datalake/curated/nyc_trips"
df_clean.write \
    .mode("overwrite") \
    .partitionBy("payment_type_str") \
    .parquet(curated_local_dir)

# Sincronizar particiones Parquet con Amazon S3 (Zona Plata: curated/nyc_trips/)
uploaded_files = 0
for root, _, files in os.walk(curated_local_dir):
    for file in files:
        if not file.endswith(".crc") and not file.startswith("_"):
            full_local_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_local_path, curated_local_dir)
            s3_key = f"curated/nyc_trips/{rel_path}".replace("\\", "/")
            s3_client.upload_file(full_local_path, S3_BUCKET, s3_key)
            uploaded_files += 1

print(f"ETL finalizado: {uploaded_files} archivos Parquet particionados subidos a s3://{S3_BUCKET}/curated/nyc_trips/")

In [ ]:
# ==============================================================================
# ANÁLISIS DESCRIPTIVO DISTRIBUIDO - EDA PYSPARK SQL (2.1.3)
# ==============================================================================
df_curated = spark.read.parquet(curated_local_dir)
df_curated.createOrReplaceTempView("vw_nyc_trips")

print("[CONSULTA PYSPARK SQL 1] Métricas Estadísticas por Método de Pago:")
sql_eda_payment = """
SELECT
    payment_type_str AS Metodo_Pago,
    COUNT(*) AS Total_Viajes,
    ROUND(AVG(trip_distance), 2) AS Distancia_Media_Mi,
    ROUND(AVG(trip_duration_min), 2) AS Duracion_Media_Min,
    ROUND(AVG(fare_amount), 2) AS Tarifa_Media_USD,
    ROUND(STDDEV(fare_amount), 2) AS Desv_Est_Tarifa_USD,
    ROUND(AVG(tip_amount), 2) AS Propina_Media_USD
FROM vw_nyc_trips
GROUP BY payment_type_str
ORDER BY Total_Viajes DESC
"""
spark.sql(sql_eda_payment).show(truncate=False)

print("[CONSULTA PYSPARK SQL 2] Demanda Horaria, Tarifas y Duración Promedio:")
sql_eda_hourly = """
SELECT
    pickup_hour AS Hora_Dia,
    COUNT(*) AS Cantidad_Viajes,
    ROUND(AVG(fare_amount), 2) AS Tarifa_Promedio_USD,
    ROUND(AVG(trip_duration_min), 2) AS Duracion_Promedio_Min
FROM vw_nyc_trips
GROUP BY pickup_hour
ORDER BY pickup_hour ASC
"""
df_hourly = spark.sql(sql_eda_hourly)
df_hourly.show(24, truncate=False)

# Visualización generada desde resultados agregados en PySpark SQL
pdf_hourly = df_hourly.toPandas()
fig, ax1 = plt.subplots(figsize=(11, 5))
sns.barplot(data=pdf_hourly, x="Hora_Dia", y="Cantidad_Viajes", color="#4C72B0", alpha=0.75, ax=ax1)
ax1.set_xlabel("Hora del Día (00:00 a 23:00 hrs)")
ax1.set_ylabel("Cantidad de Viajes", color="#4C72B0", fontweight="bold")

ax2 = ax1.twinx()
sns.lineplot(data=pdf_hourly, x="Hora_Dia", y="Duracion_Promedio_Min", color="#C44E52", marker="o", linewidth=2.5, ax=ax2)
ax2.set_ylabel("Duración Promedio del Viaje (Minutos)", color="#C44E52", fontweight="bold")

plt.title("EDA Distribuido en PySpark SQL: Demanda de Viajes vs Duración Media por Hora")
plt.tight_layout()
plt.savefig("eda_demanda_duracion.png", dpi=150)
plt.show()

In [ ]:
# ==============================================================================
# MODELO PREDICTIVO DISTRIBUIDO CON PYSPARK.ML (2.1.3)
# ==============================================================================
print("Entrenando Pipeline de Machine Learning Distribuido (RandomForestRegressor)...")

# 1. División Entrenamiento (80%) y Prueba (20%)
train_df, test_df = df_curated.randomSplit([0.8, 0.2], seed=42)

# 2. Preprocesamiento Distribuido (Imputer -> StringIndexer -> OneHotEncoder -> VectorAssembler -> StandardScaler)
imputer = Imputer(
    inputCols=["passenger_count", "RatecodeID"],
    outputCols=["passenger_count_imp", "RatecodeID_imp"]
).setStrategy("median")

indexer = StringIndexer(
    inputCol="payment_type_str",
    outputCol="payment_type_idx",
    handleInvalid="keep"
)

encoder = OneHotEncoder(
    inputCols=["payment_type_idx"],
    outputCols=["payment_type_ohe"]
)

feature_cols = [
    "trip_distance",
    "trip_duration_min",
    "passenger_count_imp",
    "RatecodeID_imp",
    "pickup_hour",
    "pickup_day_of_week",
    "payment_type_ohe"
]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")

scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withStd=True,
    withMean=False
)

# 3. Algoritmo Random Forest Regressor para predecir la tarifa ('fare_amount')
rf_model = RandomForestRegressor(
    featuresCol="features",
    labelCol="fare_amount",
    numTrees=50,
    maxDepth=8,
    seed=42
)

# 4. Ajuste del Pipeline completo
ml_pipeline = Pipeline(stages=[imputer, indexer, encoder, assembler, scaler, rf_model])
trained_model = ml_pipeline.fit(train_df)

# 5. Predicción sobre el conjunto de prueba (Test Set)
predictions = trained_model.transform(test_df)
print("\n🔍 Muestra de Predicciones vs Valores Reales (Test Set):")
predictions.select(
    "trip_distance",
    "trip_duration_min",
    "fare_amount",
    F.round("prediction", 2).alias("predicted_fare_usd")
).show(10)

# 6. Evaluación con métricas formales (RMSE, MAE, R2)
eval_rmse = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction", metricName="rmse")
eval_mae = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction", metricName="mae")
eval_r2 = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction", metricName="r2")

rmse_val = eval_rmse.evaluate(predictions)
mae_val = eval_mae.evaluate(predictions)
r2_val = eval_r2.evaluate(predictions)

print("=================================================================")
print("MÉTRICAS DE RENDIMIENTO DEL MODELO PREDICTIVO (PYSPARK.ML)")
print("=================================================================")
print(f"RMSE (Raíz del Error Cuadrático Medio): {rmse_val:.4f} USD")
print(f"MAE  (Error Absoluto Medio):            {mae_val:.4f} USD")
print(f"R²   (Coeficiente de Determinación):    {r2_val:.4f} ({r2_val*100:.2f}%)")
print("=================================================================")